# Gene Expression

This notebook demonstrates QSSA on a two-state gene expression model. The example is aligned with the biological circuit model-reduction motivation in Pandey and Murray's AutoReduce paper and the robustness analysis in the 2023 IJRNC paper.


## Model

A simple input-controlled expression model is

$$
\begin{aligned}
\dot{m} &= k_{tx}u - \gamma_m m, \\
\dot{p} &= k_{tl}m - \gamma_p p.
\end{aligned}
$$

Here $m$ is mRNA, $p$ is protein, and $u$ is an input or effective promoter activity. If mRNA dynamics are fast relative to protein dilution/degradation, QSSA sets $\dot{m}=0$.


In [ ]:
import numpy as np
from sympy import Symbol, simplify

from autoreduce.system.system import System
from autoreduce.utils.reduction import get_reducible

m = Symbol("m")
p = Symbol("p")
k_tx = Symbol("k_tx")
k_tl = Symbol("k_tl")
gamma_m = Symbol("gamma_m")
gamma_p = Symbol("gamma_p")
u = Symbol("u")

x = [m, p]
f = [
    k_tx * u - gamma_m * m,
    k_tl * m - gamma_p * p,
]

system = System(
    x,
    f,
    params=[k_tx, k_tl, gamma_m, gamma_p, u],
    params_values=[2.0, 5.0, 10.0, 0.2, 1.0],
    x_init=[0.0, 0.0],
    C=np.array([[0, 1]]),
)


## Reduce The Model

The retained state is $p$. The eliminated fast state is $m$.


In [ ]:
reducible_system = get_reducible(system)
reduced_system, collapsed_system = reducible_system.solve_timescale_separation(
    [p],
    fast_states=[m],
)

[simplify(expr) for expr in reduced_system.f]


## Interpretation

The QSSA condition gives

$$
m = \frac{k_{tx}u}{\gamma_m}.
$$

The reduced protein model is

$$
\dot{p} = \frac{k_{tl}k_{tx}u}{\gamma_m} - \gamma_p p.
$$

This reduced equation makes the effective production rate explicit.


In [ ]:
collapsed_system.x, collapsed_system.f
